# EPUB Audiobook - batch chunk synthesis (multiple patches, one run)

This notebook synthesizes the text chunks exported by the EPUB Audiobook App for
**every patch in the batch**, sequentially, using the TTS model the app selected
(read from `batch_manifest.json`). For each patch it writes `chunk_NNN.wav`
files into an `output/` subfolder inside that patch's folder, and as soon as a
patch is complete it merges the chunks into a single **`result/NNN - <patch name>.wav`**
at the batch root.

Supported models: **voxcpm2** / **omnivoice** / **vieneu-fast** (local, on a GPU,
cloning the shared voice reference clip) and **edge-tts** / **gTTS** (online, no
GPU, using a `voice_id` from the app instead of a reference clip).

> **Set `IS_KAGGLE` first.** Cell 1 defines one global flag, `IS_KAGGLE`, that
> every cell in this notebook uses — there is **no per-cell auto-detection**.
> Set it to `True` when running on Kaggle or `False` when running on Google
> Colab, then run the cells top to bottom.

> **Enable a GPU for the offline models.** `voxcpm2`/`omnivoice`/`vieneu-fast`
> run on CUDA. In Colab: **Runtime > Change runtime type > GPU (T4)**, then
> restart the session (GPU, not TPU). `edge-tts`/`gTTS` skip this check entirely.

It reads everything it needs from `batch_manifest.json`, which was exported
alongside this notebook - you should not need to type any patch info by hand.

## No re-downloading the model on restart
Cell 1 points the Hugging Face cache at persistent storage, so the multi-GB
offline model weights are downloaded **once**, not on every session:

- **Colab**: cached in your Drive at `EPUB Audiobook Exports/.cache` (the
  first run is a normal download that lands in Drive; later sessions load
  from there). Uses a few GB of Drive space.
- **Kaggle**: cached in `/kaggle/working/.cache` - turn on
  **Persistence: Files only** in the notebook options so it survives new
  sessions.

## Disconnects are fine - just re-run
Free Colab/Kaggle sessions can die mid-run. The synthesis cell is safe to
re-run any number of times:

- chunks that already have a `.wav` are **skipped**,
- patches that already have their merged `result/` file are **skipped entirely**,
- on Colab every `.wav` is written **directly into your Drive folder**, so
  progress survives even if the runtime is killed - reconnect, run the cells
  top to bottom (the model reloads), and it continues where it stopped.

## Google Colab (recommended)
Keep `IS_KAGGLE = False` in Cell 1. The app already uploaded this folder into
**your own Google Drive** (the account you connected). Just run the cells top
to bottom: cell 3 mounts your Drive, and the folder is a normal filesystem
path from then on - no Google API calls needed in this notebook at all. Merged
patch files end up in the `result/` subfolder of the batch folder in Drive.

## Kaggle (no setup - the Drive credentials are already in Cell 4)
Set `IS_KAGGLE = True` in Cell 1. Kaggle has no native Google Drive mount, but
Cell 4 talks to the Drive API directly using the app's own credentials, giving
Kaggle the same experience as Colab. The app wrote the credentials of the
exporting account into `GDRIVE_CREDS` in Cell 4 when it built this notebook, so
there is nothing to configure - just run the cells top to bottom.

> **Keep this notebook private.** `GDRIVE_CREDS` is a live key (refresh token +
> client secret) to the Google account that owns the export. Anyone who gets the
> `.ipynb` can read and write the app's Drive files, so do not publish the
> notebook on Kaggle or pass the file around.

If you'd rather not carry the credentials inside the file, either of these still
works:

- paste them yourself - in the app open the **Google Drive** page, click
  **Copy Kaggle credentials**, and put that JSON into `GDRIVE_CREDS` in Cell 4;
- or clear `GDRIVE_CREDS` and store the same JSON as a Kaggle secret named
  **`GDRIVE_CREDS`** (**Add-ons > Secrets**, enabled for this notebook) - Cell 4
  falls back to the secret whenever `GDRIVE_CREDS` is empty.

Cell 4 then indexes the batch folder on Drive and downloads only the manifests
and the shared voice reference into `/kaggle/working`. Chunk `.wav` files are
fetched on demand at merge time, and finished `result/` files are never
downloaded at all, so restarting a session costs a few small files instead of
the whole batch. Cell 8 uploads every generated `.wav` and merged `result/`
file **straight back to Drive** as it is written. A dead Kaggle session resumes exactly like
Colab - re-run the cells and it continues - and the app can import the
results from Drive as usual.

### Kaggle without Drive (zip-dataset fallback)
If you'd rather not store credentials on Kaggle (still set `IS_KAGGLE = True`):
1. In the app, use **Download selected (.zip)**, upload the zip as a Kaggle
   Dataset and attach it to this notebook.
2. Skip Cells 3 and 4 and set `FOLDER_PATH` to the dataset path
   (e.g. `/kaggle/input/<dataset-name>`) - see the comment in Cell 4.
3. Output goes under `/kaggle/working/result/`; download it from Kaggle's
   **Output** pane, which offers its own "Download All".
4. **Resuming across sessions:** `/kaggle/working` does not survive a new
   session. To resume, add the `.wav` files you already produced to the
   dataset under `patches/patch_NNN/output/` - the skip check looks there
   too and will not redo those chunks.


In [ ]:
# Cell 1: platform flag + persistent caches + dependencies.
#
# >>> SET THIS FIRST <<<
# IS_KAGGLE is the single global switch used by EVERY cell in this notebook -
# there is no per-cell auto-detection. Set it before running anything:
#   True  -> running on Kaggle
#   False -> running on Google Colab
IS_KAGGLE = False

# Persistent caches: point the Hugging Face model cache (and pip's download
# cache) at persistent storage, so restarting the session does NOT re-download
# the multi-GB model weights every time:
#   - Colab: cached in your Google Drive under "EPUB Audiobook Exports/.cache".
#     The first run downloads the model once into Drive; every later session
#     loads it from there.
#   - Kaggle: cached in /kaggle/working/.cache. Enable the notebook's
#     "Persistence: Files only" setting (right sidebar > Notebook options) so
#     the cache survives across sessions - without it the cache still helps
#     within one session, but a brand-new session starts empty.
import os

USE_PERSISTENT_CACHE = True  # set False to use the default ephemeral cache

CACHE_ROOT = None
if USE_PERSISTENT_CACHE:
    if IS_KAGGLE:
        CACHE_ROOT = "/kaggle/working/.cache"
    else:
        try:
            from google.colab import drive
            DRIVE_MOUNT = "/content/drive"
            if os.path.isdir(DRIVE_MOUNT) and os.listdir(DRIVE_MOUNT) and not os.path.isdir(os.path.join(DRIVE_MOUNT, "MyDrive")):
                # Colab refuses to mount over a non-empty directory. This can be left
                # behind by an interrupted mount or by another notebook cell.
                DRIVE_MOUNT = "/content/epub_audiobook_drive"
            if not os.path.isdir(os.path.join(DRIVE_MOUNT, "MyDrive")):
                drive.mount(DRIVE_MOUNT)
            CACHE_ROOT = os.path.join(DRIVE_MOUNT, "MyDrive/EPUB Audiobook Exports/.cache")
        except Exception as exc:
            print(f"Drive not available ({type(exc).__name__}) - using ephemeral cache.")

if CACHE_ROOT:
    os.makedirs(CACHE_ROOT, exist_ok=True)
    # Must be set BEFORE anything imports huggingface_hub (it reads HF_HOME at
    # import time), which is why this cell comes first.
    os.environ["HF_HOME"] = os.path.join(CACHE_ROOT, "huggingface")
    # Drive's FUSE mount doesn't support symlinks; the HF cache detects that and
    # falls back to plain file copies - silence the warning about it.
    os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
    os.environ["PIP_CACHE_DIR"] = os.path.join(CACHE_ROOT, "pip")
    print("Persistent cache:", CACHE_ROOT)
else:
    print("No persistent cache - the model will be re-downloaded each session.")

# Base deps used by every model plus the online models (edge-tts / gTTS). The
# heavy per-model packages are pip-installed lazily in Cell 7 only when needed.
!pip install -q soundfile numpy edge-tts gTTS


In [ ]:
# Cell 2: (optional) Hugging Face token - avoids the "unauthenticated requests" rate
# limit warning/slow downloads when fetching the model. Get a free token at
# https://huggingface.co/settings/tokens
#
# Recommended: store it as a secret instead of pasting it in plain text here -
# Colab: left sidebar > key icon > add secret named HF_TOKEN.
# Kaggle: Add-ons > Secrets > add secret named HF_TOKEN.
# If no secret is found, you'll get a hidden prompt to paste it manually (or just
# press Enter to skip and continue unauthenticated).
# The app replaces __HF_TOKEN__ below with the token from its own settings on export;
# leave the app's HF_TOKEN setting empty to keep this as a placeholder (secrets/prompt).
# Which secret store is read follows the global IS_KAGGLE flag from Cell 1.
HF_TOKEN = "__HF_TOKEN__"

if not HF_TOKEN:
    if IS_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN") or ""
        except Exception:
            pass
    else:
        try:
            from google.colab import userdata
            HF_TOKEN = userdata.get("HF_TOKEN") or ""
        except Exception:
            pass

if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass("Hugging Face token (leave blank to skip): ")

if HF_TOKEN:
    import os

    # Deliberately NOT huggingface_hub.login(): that import pulls in typer, which
    # needs click >= 8.2, while Kaggle's image still ships click 8.1.x - the import
    # dies with "TypeError: type 'Choice' is not subscriptable" before it ever gets
    # to the token. Setting the env vars is what login() effectively does for API
    # access anyway: the hub client reads them on every request. Both names are set
    # because older packages still look for HUGGING_FACE_HUB_TOKEN.
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

    # HfApi lives in a module that does not import typer, so this check is safe on
    # both platforms. A bad token must not abort the run - unauthenticated
    # downloads still work, just rate limited.
    try:
        from huggingface_hub import HfApi

        print("Authenticated to Hugging Face Hub as:", HfApi().whoami(token=HF_TOKEN)["name"])
    except Exception as exc:
        print(f"HF token set, but could not verify it ({type(exc).__name__}: {exc}).")
        print("Continuing - downloads will still be attempted with this token.")
else:
    print("No HF token set - continuing unauthenticated (may hit rate limits).")


In [ ]:
# Cell 3: Google Colab only - mount your Drive. The exported batch folder is located
# automatically by batch id, so you do NOT need to paste the folder name by hand.
# Uses the global IS_KAGGLE flag from Cell 1: on Kaggle this cell does nothing
# (Cell 4 takes over), so "Run all" works on both platforms.
import glob, json, os

if IS_KAGGLE:
    print("IS_KAGGLE = True - skipping the Colab Drive mount (Cell 4 downloads the batch instead).")
else:
    from google.colab import drive

    DRIVE_MOUNT = globals().get("DRIVE_MOUNT", "/content/drive")
    if not os.path.isdir(os.path.join(DRIVE_MOUNT, "MyDrive")):
        drive.mount(DRIVE_MOUNT)

    BATCH_ID = "__BATCH_ID__"  # injected by the app when this notebook was exported
    EXPORTS_ROOT = os.path.join(DRIVE_MOUNT, "MyDrive/EPUB Audiobook Exports")
    DEFAULT_FOLDER = os.path.join(EXPORTS_ROOT, "__DEFAULT_FOLDER_NAME__")

    def _read_batch_id(folder):
        """Return the batch_id inside folder/batch_manifest.json, or a short reason string."""
        manifest_path = os.path.join(folder, "batch_manifest.json")
        if not os.path.isfile(manifest_path):
            return "<no batch_manifest.json>"
        try:
            with open(manifest_path, encoding="utf-8") as f:
                return json.load(f).get("batch_id")
        except Exception as exc:
            return f"<unreadable batch_manifest.json: {exc}>"

    # Scan every export folder and match the one whose batch_manifest.json has our batch id.
    FOLDER_PATH = None
    for d in sorted(glob.glob(os.path.join(EXPORTS_ROOT, "*")), reverse=True):
        if os.path.isdir(d) and _read_batch_id(d) == BATCH_ID:
            FOLDER_PATH = d
            break

    if FOLDER_PATH is None:
        FOLDER_PATH = DEFAULT_FOLDER  # fall back to the exact name the app used at export time

    print("Using folder:", FOLDER_PATH)
    if not os.path.isdir(FOLDER_PATH):
        # Folder check: list what actually is under the exports root so a wrong Google
        # account or an incomplete rclone push is obvious at a glance.
        print(f"\nbatch_id {BATCH_ID} not found. Export folders under {EXPORTS_ROOT}:")
        _visible = [d for d in sorted(glob.glob(os.path.join(EXPORTS_ROOT, "*"))) if os.path.isdir(d)]
        if _visible:
            for d in _visible:
                print(f"  - {os.path.basename(d)}  (batch_id={_read_batch_id(d)})")
        else:
            print(f"  (nothing - {EXPORTS_ROOT} is empty or missing on the account you mounted)")
        print(
            "\nIf the folder is missing above: you mounted a different Google account than the "
            "one the batch was exported to, or the rclone push has not finished. Mount the "
            "account that owns the batch, or set FOLDER_PATH manually to one of the folders listed."
        )
    assert os.path.isdir(FOLDER_PATH), f"Folder not found: {FOLDER_PATH}"


In [ ]:
# Cell 4: Kaggle only - connect to Google Drive with the GDRIVE_CREDS credentials and
# download the exported batch folder. Uses the global IS_KAGGLE flag from Cell 1:
# on Colab this cell does nothing (Cell 3 already mounted Drive), so "Run all"
# works on both platforms.
#
# Credentials: the app bakes the JSON of the exporting account straight into
# GDRIVE_CREDS below, so there is nothing to set up - just run the cell. You can also
# paste it by hand (Google Drive page > "Copy Kaggle credentials"), and if you leave
# GDRIVE_CREDS empty the cell falls back to a Kaggle secret named GDRIVE_CREDS
# (Add-ons > Secrets), which is how earlier exports worked.
#
# >>> These credentials are a live Drive key (refresh token + client secret) for the
# >>> account that owns the export. Keep this notebook PRIVATE on Kaggle and do not
# >>> share the .ipynb - anyone holding it can read and write the app's Drive files.
# >>> Prefer the secret if the notebook may end up public: clear GDRIVE_CREDS below.
#
# This cell indexes the batch folder on Drive and downloads only the manifests and the
# shared voice reference into /kaggle/working/batch. Chunk .wav files are fetched on
# demand by Cell 8, and result .wav files are never downloaded. It also defines
# drive_persist(), which Cell 8 uses to upload every generated .wav and merged result
# file straight back to Drive.
#
# --- zip-dataset fallback (no Drive credentials) ---
# Skip this cell too and point FOLDER_PATH at the attached dataset instead. Use the
# EXACT zip filename (without .zip) as the dataset name when uploading, e.g.:
# FOLDER_PATH = "/kaggle/input/<dataset-name>"
import io
import json
import os
import threading
from concurrent.futures import ThreadPoolExecutor

# BEGIN CELL 4 HELPERS
def plan_batch_downloads(batch_manifest, remote_files):
    """Batch-relative paths worth downloading before synthesis starts.

    Only the batch manifest, each patch manifest and the shared reference clip are
    read up front. Result WAVs are never read by this notebook at all - Cell 8 only
    tests whether they exist, which the inventory answers. Chunk WAVs are fetched
    on demand at merge time, and anything else an older batch folder happens to
    hold (backgrounds, music) is ignored."""
    wanted = ["batch_manifest.json"]
    reference = batch_manifest.get("reference_wav")
    if reference:
        wanted.append(reference)
    for entry in batch_manifest.get("patches", []):
        wanted.append(f"{entry['folder']}/manifest.json")
    return [rel for rel in wanted if rel in remote_files]
# END CELL 4 HELPERS

if not IS_KAGGLE:
    print("IS_KAGGLE = False - skipping the Kaggle Drive download (Cell 3 already mounted Drive).")
else:
    !pip install -q google-api-python-client google-auth

    from google.oauth2.credentials import Credentials
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload

    BATCH_ID = "__BATCH_ID__"  # injected by the app when this notebook was exported

    # Injected by the app on export (see the warning at the top of this cell); paste
    # your own "Copy Kaggle credentials" JSON here to override it, or blank it out to
    # go back to the Kaggle secret.
    GDRIVE_CREDS = "__GDRIVE_CREDS__"

    _creds_raw = GDRIVE_CREDS.strip()
    if _creds_raw.startswith("__") and _creds_raw.endswith("__"):
        _creds_raw = ""  # untouched placeholder: this notebook was never run through the app
    if not _creds_raw:
        # No inline credentials - fall back to the Kaggle secret.
        from kaggle_secrets import UserSecretsClient

        _creds_raw = UserSecretsClient().get_secret("GDRIVE_CREDS") or ""
    assert _creds_raw, (
        "No Drive credentials. Either re-export the batch from the app (which fills in "
        "GDRIVE_CREDS above), paste the JSON from the app's Google Drive page > "
        "\"Copy Kaggle credentials\", or add it as a Kaggle secret named GDRIVE_CREDS."
    )
    try:
        creds_info = json.loads(_creds_raw)
    except ValueError as exc:
        raise AssertionError(
            f"GDRIVE_CREDS is not valid JSON ({exc}). Copy it again from the app's "
            "Google Drive page - it must be the whole {...} object."
        ) from exc
    creds = Credentials(
        token=None,
        refresh_token=creds_info["refresh_token"],
        token_uri="https://oauth2.googleapis.com/token",
        client_id=creds_info["client_id"],
        client_secret=creds_info["client_secret"],
        scopes=["https://www.googleapis.com/auth/drive.file"],
    )
    drive_service = build("drive", "v3", credentials=creds)

    FOLDER_MIME = "application/vnd.google-apps.folder"


    def _list_children(folder_id):
        files, token = [], None
        while True:
            resp = drive_service.files().list(
                q=f"'{folder_id}' in parents and trashed = false",
                fields="nextPageToken, files(id, name, mimeType)",
                pageToken=token, pageSize=1000,
            ).execute()
            files += resp.get("files", [])
            token = resp.get("nextPageToken")
            if not token:
                return files


    def _download(file_id, dest):
        request = drive_service.files().get_media(fileId=file_id)
        downloader = MediaIoBaseDownload(dest, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()


    # Locate the batch folder: scan "EPUB Audiobook Exports" for the folder whose
    # batch_manifest.json carries this notebook's batch id.
    resp = drive_service.files().list(
        q=f"name = 'EPUB Audiobook Exports' and mimeType = '{FOLDER_MIME}' and trashed = false",
        fields="files(id)",
    ).execute()
    _roots = resp.get("files", [])
    assert _roots, (
        "No 'EPUB Audiobook Exports' folder found. The credentials only see files "
        "created by the app itself - make sure you exported this batch to Drive."
    )

    _batch_folder_id = None
    _seen_folders = []  # (name, batch_id) for every export folder these creds can see
    for _root in _roots:
        for _folder in _list_children(_root["id"]):
            if _folder["mimeType"] != FOLDER_MIME:
                continue
            _mf = next((f for f in _list_children(_folder["id"]) if f["name"] == "batch_manifest.json"), None)
            if _mf is None:
                _seen_folders.append((_folder["name"], "<no batch_manifest.json>"))
                continue
            _buf = io.BytesIO()
            _download(_mf["id"], _buf)
            _found_id = json.loads(_buf.getvalue().decode("utf-8")).get("batch_id")
            _seen_folders.append((_folder["name"], _found_id))
            if _found_id == BATCH_ID:
                _batch_folder_id = _folder["id"]
                print("Found batch folder on Drive:", _folder["name"])
                break
        if _batch_folder_id:
            break

    if not _batch_folder_id:
        # Folder check: show exactly what these credentials CAN see, so a mismatch is
        # obvious. The drive.file scope only reveals folders THIS OAuth client created,
        # so a batch pushed via rclone or exported to a different Google account is
        # invisible here even though it exists on Drive.
        print(f"\nbatch_id {BATCH_ID} not found. Export folders these credentials can see:")
        if _seen_folders:
            for _name, _bid in _seen_folders:
                print(f"  - {_name}  (batch_id={_bid})")
        else:
            print("  (none - this account has no app-created export folders)")
        raise AssertionError(
            f"No Drive folder found with batch_id {BATCH_ID}. If it is missing above, the "
            "batch was pushed via rclone or to a different Google account (the drive.file "
            "scope cannot see either). Use the account that owns the batch, or fall back to "
            "the zip-dataset method: skip this cell and set FOLDER_PATH = /kaggle/input/<dataset-name>."
        )

    # Listing is separate from downloading. We already have to walk the folder to learn
    # every file id for drive_persist(), and that walk doubles as the remote inventory:
    # Cell 8 decides what is already merged or synthesized from it without transferring
    # a byte. Result WAVs and chunk WAVs are never downloaded here.
    FOLDER_PATH = "/kaggle/working/batch"
    _drive_folder_ids = {"": _batch_folder_id}
    _drive_file_ids = {}


    def _list_tree(folder_id, rel):
        for f in _list_children(folder_id):
            child_rel = f"{rel}/{f['name']}" if rel else f["name"]
            if f["mimeType"] == FOLDER_MIME:
                _drive_folder_ids[child_rel] = f["id"]
                _list_tree(f["id"], child_rel)
            else:
                _drive_file_ids[child_rel] = f["id"]


    _list_tree(_batch_folder_id, "")
    print(f"Indexed {len(_drive_file_ids)} files on Drive (nothing downloaded yet).")

    # googleapiclient's http object is NOT thread safe - sharing one service across a
    # pool produces intermittent, confusing failures rather than clean errors. Give each
    # worker thread its own service; build() uses static discovery, so it is cheap and
    # makes no network call.
    _thread_local = threading.local()


    def _service():
        service = getattr(_thread_local, "service", None)
        if service is None:
            service = build("drive", "v3", credentials=creds)
            _thread_local.service = service
        return service


    def _download_to(rel, dest):
        if rel not in _drive_file_ids:
            raise RuntimeError(f"not on Drive: {rel}")
        os.makedirs(os.path.dirname(dest) or ".", exist_ok=True)
        request = _service().files().get_media(fileId=_drive_file_ids[rel])
        tmp = dest + ".part"
        with open(tmp, "wb") as fh:
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done:
                _, done = downloader.next_chunk()
        os.replace(tmp, dest)
        return dest


    def drive_fetch_many(pairs):
        """Download [(batch_relative_path, local_destination)] in parallel.

        Raises on the first failure; callers treat that as a resumable condition
        rather than aborting the batch."""
        pairs = list(pairs)
        if not pairs:
            return []
        with ThreadPoolExecutor(max_workers=8) as pool:
            return list(pool.map(lambda pair: _download_to(*pair), pairs))


    def drive_fetch(rel, dest=None):
        """Download one batch-relative path into the local batch folder."""
        return drive_fetch_many([(rel, dest or os.path.join(FOLDER_PATH, rel))])[0]


    # Bootstrap: the batch manifest names the patch folders, so it comes down first.
    drive_fetch("batch_manifest.json")
    with open(os.path.join(FOLDER_PATH, "batch_manifest.json"), encoding="utf-8") as fh:
        _bootstrap_manifest = json.load(fh)

    _wanted = plan_batch_downloads(_bootstrap_manifest, _drive_file_ids)
    _pending = [
        (rel, os.path.join(FOLDER_PATH, rel))
        for rel in _wanted
        if not os.path.exists(os.path.join(FOLDER_PATH, rel))
    ]
    drive_fetch_many(_pending)
    print(f"Batch ready at {FOLDER_PATH} ({len(_pending)} file(s) downloaded).")


    def drive_persist(local_path, rel_dir):
        """Upload a freshly written file into rel_dir inside the batch folder on Drive,
        creating the subfolder chain as needed and replacing an existing file in place.
        Cell 8 calls this after every chunk .wav and every merged result file, so a dead
        Kaggle session loses nothing."""
        parent, rel = _drive_folder_ids[""], ""
        for part in [p for p in rel_dir.split("/") if p]:
            rel = f"{rel}/{part}" if rel else part
            if rel not in _drive_folder_ids:
                folder = drive_service.files().create(
                    body={"name": part, "mimeType": FOLDER_MIME, "parents": [parent]},
                    fields="id",
                ).execute()
                _drive_folder_ids[rel] = folder["id"]
            parent = _drive_folder_ids[rel]
        name = os.path.basename(local_path)
        file_rel = f"{rel}/{name}" if rel else name
        media = MediaFileUpload(local_path)
        if file_rel in _drive_file_ids:
            drive_service.files().update(fileId=_drive_file_ids[file_rel], media_body=media).execute()
        else:
            created = drive_service.files().create(
                body={"name": name, "parents": [parent]}, media_body=media, fields="id",
            ).execute()
            _drive_file_ids[file_rel] = created["id"]


In [ ]:
# Cell 5: load the batch manifest and resolve which TTS model to run.
# Generic manifest contract:
#   batch_manifest["tts"] = {"model_id": <id>, "options": {...}, "voice_id": <voice>}
# Supported model ids: voxcpm2, omnivoice, vieneu-fast, edge-tts, gtts.
# Legacy fallback: batch_manifest["voxcpm_model_id"] (always openbmb/VoxCPM2).
import json
import os

with open(os.path.join(FOLDER_PATH, "batch_manifest.json"), "r", encoding="utf-8") as f:
    batch_manifest = json.load(f)

print(f"Batch of {batch_manifest['patch_count']} patches from book '{batch_manifest['book_title']}'")
for entry in batch_manifest["patches"]:
    print(f"  patch {entry['patch_index']:03d}: {entry['patch_name']} "
          f"(chapters {entry['chapter_start']}-{entry['chapter_end']}, {entry['chunk_count']} chunks)")

# --- model selection ---
TTS_CFG = batch_manifest.get("tts") or {}
TTS_MODEL = TTS_CFG.get("model_id")
if not TTS_MODEL:
    TTS_MODEL = "voxcpm2" if batch_manifest.get("voxcpm_model_id") else "voxcpm2"
TTS_OPTIONS = TTS_CFG.get("options") or {}
VOICE_ID = TTS_CFG.get("voice_id") or batch_manifest.get("voice_id")
print("TTS model:", TTS_MODEL)

# The offline cloning models need a voice reference clip; edge-tts/gTTS ignore it.
REFERENCE_MODELS = {"voxcpm2", "omnivoice", "vieneu-fast"}
REFERENCE_REQUIRED = TTS_MODEL in REFERENCE_MODELS

reference_wav_path = None
prompt_text = None
if batch_manifest.get("reference_wav"):
    reference_wav_path = os.path.join(FOLDER_PATH, batch_manifest["reference_wav"])
    prompt_text = batch_manifest.get("reference_transcript") or None

if REFERENCE_REQUIRED:
    if not reference_wav_path or not os.path.exists(reference_wav_path):
        raise RuntimeError(
            "Voice reference clip not found in this export - it is required so every "
            "chunk is synthesized with the same voice. In the app, upload a voice "
            "reference clip for this book, then re-export the batch."
        )
    print(f"Using cloned voice reference: {reference_wav_path}")
else:
    print(f"{TTS_MODEL} synthesizes online and does not need a voice reference clip.")

# The online models need a voice id instead of a reference clip.
if TTS_MODEL in ("edge-tts", "gtts"):
    if not VOICE_ID:
        raise RuntimeError(
            f"{TTS_MODEL} requires batch_manifest['tts']['voice_id'] "
            "(edge-tts: e.g. 'en-US-AriaNeural'; gTTS: e.g. 'en'). Re-export the "
            "batch with a voice selected."
        )
    print("Using voice:", VOICE_ID)

# Fixed sample rates per model; edge-tts and gTTS both synthesize at 24 kHz.
SAMPLE_RATE = {
    "voxcpm2": 16000,
    "omnivoice": 24000,
    "vieneu-fast": 48000,
    "edge-tts": 24000,
    "gtts": 24000,
}.get(TTS_MODEL, 16000)
print("Sample rate:", SAMPLE_RATE)


In [ ]:
# Cell 6: check you're actually on a GPU - but ONLY for the offline models
# (voxcpm2 / omnivoice / vieneu-fast are far too slow on CPU). edge-tts/gTTS
# synthesize online and skip this check entirely, so they run on CPU-only
# Colab/Kaggle sessions too.
GPU_MODELS = {"voxcpm2", "omnivoice", "vieneu-fast"}

if TTS_MODEL in GPU_MODELS:
    import torch

    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        raise RuntimeError(
            f"{TTS_MODEL} needs a GPU - Colab: Runtime > Change runtime type > "
            "GPU (T4), then Restart session (choose GPU, not TPU). On Kaggle: "
            "enable a GPU accelerator in the sidebar."
        )
else:
    print(f"{TTS_MODEL} runs online - no GPU required.")


In [ ]:
# Cell 7: load the model for the resolved TTS_MODEL. edge-tts/gTTS are online and
# need no model download; the heavy packages are pip-installed here, lazily, only
# when the manifest selects that model.
import os
import subprocess
import sys

if TTS_MODEL in ("edge-tts", "gtts"):
    print(f"{TTS_MODEL} is online - no model to download.")
    model = None
else:
    package = {
        "voxcpm2": "voxcpm",
        "omnivoice": "omnivoice",
        "vieneu-fast": "vieneu",
    }.get(TTS_MODEL)
    if package:
        os.system(f"{sys.executable} -m pip install -q {package}")
    if TTS_MODEL == "voxcpm2":
        from voxcpm import VoxCPM

        model = VoxCPM.from_pretrained(
            TTS_OPTIONS.get("model_id") or "openbmb/VoxCPM2",
            load_denoiser=bool(TTS_OPTIONS.get("load_denoiser", False)),
        )
    elif TTS_MODEL == "omnivoice":
        import torch
        from omnivoice import OmniVoice

        device = TTS_OPTIONS.get("device") or ("cuda" if torch.cuda.is_available() else "cpu")
        dtype = torch.float16 if device.startswith("cuda") else torch.float32
        model = OmniVoice.from_pretrained(
            TTS_OPTIONS.get("model_id") or "k2-fsa/OmniVoice",
            device_map=device,
            dtype=dtype,
        )
    elif TTS_MODEL == "vieneu-fast":
        from vieneu import Vieneu

        model = Vieneu(precision=TTS_OPTIONS.get("precision", "int8"))
    else:
        raise RuntimeError(f"Unsupported TTS model: {TTS_MODEL}")
    print("Loaded model:", type(model).__name__)


In [ ]:
# Cell 8: synthesize every patch in order, merging each patch into result/ as soon
# as it completes. Safe to re-run after any disconnect:
#   - chunks with an existing .wav are skipped,
#   - patches with an existing merged result file are skipped entirely,
#   - on Colab everything below is written straight into the Drive-mounted folder,
#     so progress is persisted the moment each file is written.
import json
import os

import soundfile as sf
import tempfile
import warnings

# BEGIN CELL 8 HELPERS
import os, tempfile, warnings
import json
import soundfile as sf
def validate_chunk_metadata(metadata, chunks):
    required = {"filename", "chapter_index", "chapter_title", "is_chapter_start", "text"}
    if not isinstance(metadata, list) or len(metadata) != len(chunks) or not metadata:
        return None
    previous_index = None
    current_index = None
    for offset, item in enumerate(metadata):
        if not isinstance(item, dict) or set(item) != required:
            return None
        if type(item["filename"]) is not str or item["filename"] != chunks[offset]:
            return None
        if type(item["chapter_index"]) is not int or type(item["chapter_title"]) is not str or not item["chapter_title"].strip() or type(item["is_chapter_start"]) is not bool:
            return None
        if type(item["text"]) is not str or not item["text"].strip():
            return None
        index = item["chapter_index"]
        if previous_index is not None and index < previous_index:
            return None
        if offset == 0 and not item["is_chapter_start"]:
            return None
        if item["is_chapter_start"]:
            if current_index == index or (previous_index is not None and index == previous_index):
                return None
            current_index = index
        elif current_index != index:
            return None
        previous_index = index
    return metadata

def merge_wav_files(paths, result_path, pause_ms, metadata=None, chunk_names=None):
    headers = []
    try:
        for path in paths:
            with sf.SoundFile(path) as wav:
                header = (wav.samplerate, wav.channels, wav.frames)
            if headers and header[:2] != headers[0][:2]:
                raise RuntimeError("chunk WAV sample rates/channels do not match")
            headers.append(header)
    except Exception as exc:
        raise RuntimeError(f"chunk WAV preflight failed: {exc}") from exc
    sample_rate, channels = headers[0][:2]
    pause_frames = round(sample_rate * pause_ms / 1000)
    import numpy as np
    silence = np.zeros((pause_frames, channels), dtype="int16") if channels > 1 else np.zeros(pause_frames, dtype="int16")
    result_tmp = tempfile.NamedTemporaryFile(prefix=".result-", suffix=".wav", dir=os.path.dirname(result_path), delete=False).name
    try:
        with sf.SoundFile(result_tmp, mode="w", samplerate=sample_rate, channels=channels, format="WAV", subtype="PCM_16") as out:
            for offset, path in enumerate(paths):
                if offset: out.write(silence)
                with sf.SoundFile(path) as inp:
                    while True:
                        block = inp.read(65536, dtype="int16", always_2d=(channels > 1))
                        if not len(block): break
                        out.write(block)
        os.replace(result_tmp, result_path)
    finally:
        if os.path.exists(result_tmp):
            try: os.unlink(result_tmp)
            except OSError: pass
    valid = validate_chunk_metadata(metadata, chunk_names or [os.path.basename(path) for path in paths]) if metadata is not None else None
    if metadata is None: warnings.warn("chunk_metadata missing; merged without timeline")
    if metadata is not None and valid is None: warnings.warn("Invalid chunk_metadata; merged without timeline")
    if valid is None: return None
    total_frames = sum(header[2] for header in headers) + pause_frames * max(0, len(headers) - 1)
    chapters, position = [], 0
    for item, header in zip(valid, headers):
        if item["is_chapter_start"]:
            chapters.append({"chapter_index": item["chapter_index"], "title": item["chapter_title"], "start_frame": position, "start_seconds": position / sample_rate})
        position += header[2] + pause_frames
    return {"version": 1, "sample_rate": sample_rate, "total_frames": total_frames, "chapters": chapters}

def write_timeline_atomic(timeline, timeline_path):
    side_tmp = tempfile.NamedTemporaryFile(prefix=".timeline-", suffix=".json", dir=os.path.dirname(timeline_path), mode="w", encoding="utf-8", delete=False)
    try:
        json.dump(timeline, side_tmp); side_tmp.flush(); os.fsync(side_tmp.fileno()); side_tmp.close(); os.replace(side_tmp.name, timeline_path)
    finally:
        try:
            if not side_tmp.closed: side_tmp.close()
            if os.path.exists(side_tmp.name): os.unlink(side_tmp.name)
        except OSError: pass

def chunk_text_for(manifest, offset, patch_dir):
    """Chunk text from the manifest, falling back to chunk_NNN.txt on disk.

    The fallback exists only for the zip-dataset path, where an older package is
    attached whole as a Kaggle dataset. On the Drive path Cell 4 never downloads
    chunk_NNN.txt, so there is nothing to fall back to."""
    metadata = manifest.get("chunk_metadata")
    if isinstance(metadata, list) and offset < len(metadata):
        text = metadata[offset].get("text")
        if isinstance(text, str) and text.strip():
            return text
    with open(os.path.join(patch_dir, manifest["chunks"][offset]), "r", encoding="utf-8") as f:
        return f.read()

def available_wavs(dirs, remote_files, remote_dir):
    """Set of .wav basenames present in any of dirs locally, or on Drive directly
    under remote_dir. One listdir per directory replaces two stats per chunk - on
    Colab those stats are FUSE round trips."""
    names = set()
    for directory in dirs:
        if os.path.isdir(directory):
            names.update(n for n in os.listdir(directory) if n.endswith(".wav"))
    prefix = remote_dir.rstrip("/") + "/"
    for rel in remote_files:
        if rel.startswith(prefix) and rel.endswith(".wav"):
            tail = rel[len(prefix):]
            if "/" not in tail:
                names.add(tail)
    return names
def save_online_mp3(text, voice_id, model_id, dest):
    if model_id == "edge-tts":
        import asyncio
        import edge_tts

        async def _go():
            await edge_tts.Communicate(text, voice_id).save(dest)
        asyncio.run(_go())
    else:  # gtts
        from gtts import gTTS

        gTTS(text=text, lang=voice_id).save(dest)


def mp3_to_wav(src, dest):
    data, _rate = sf.read(src)
    sf.write(dest, data, SAMPLE_RATE)
    os.remove(src)

def normalize_chunk_manifest(manifest):
    """Rebuild what a compact manifest leaves out - the chunks and
    expected_outputs lists, plus each entry's filename and chapter_title (the
    latter from the chapter_titles map). Returns False when the manifest says
    nothing usable about how many chunks it has, so the caller can fail that one
    patch instead of the whole batch. Older manifests carry every field already
    and pass through untouched."""
    if not isinstance(manifest.get("chunks"), list):
        count = manifest.get("chunk_count")
        if not isinstance(count, int) or isinstance(count, bool) or count < 1:
            return False
        manifest["chunks"] = [f"chunk_{i:03d}.txt" for i in range(count)]
    if not isinstance(manifest.get("expected_outputs"), list):
        manifest["expected_outputs"] = [f"chunk_{i:03d}.wav" for i in range(len(manifest["chunks"]))]
    titles = manifest.get("chapter_titles") or {}
    for offset, item in enumerate(manifest.get("chunk_metadata") or []):
        if not isinstance(item, dict):
            continue
        if "filename" not in item and offset < len(manifest["chunks"]):
            item["filename"] = manifest["chunks"][offset]
        if "chapter_title" not in item:
            item["chapter_title"] = titles.get(str(item.get("chapter_index")))
    return True

# END CELL 8 HELPERS

# --- config ---
PATCH_IDS = None      # None = run ALL patches in the batch; or e.g. [12, 15] to restrict
SKIP_EXISTING = True  # skip chunks/patches that already have output (safe resume)
_CHUNK_PAUSE_MS = 300

# Drive upload hook: defined by the Kaggle Drive cell (Cell 4); a no-op everywhere
# else - on Colab files are already written straight into the Drive mount, and the
# zip-dataset fallback has nowhere to upload to.
persist = globals().get("drive_persist") or (lambda local_path, rel_dir: None)

# Remote inventory built by Cell 4 (Kaggle Drive mode). Empty on Colab and in the
# zip-dataset fallback, so both platforms take one code path below.
REMOTE = globals().get("_drive_file_ids") or {}

def _no_fetch(pairs):
    raise RuntimeError(
        "chunk WAVs exist only on Drive but this session has no Drive connection"
    )

drive_fetch_many = globals().get("drive_fetch_many") or _no_fetch

# /kaggle/input is a read-only mount, so with the zip-dataset fallback all output
# goes under /kaggle/working instead. Otherwise FOLDER_PATH is writable (the Drive
# mount on Colab, or /kaggle/working/batch in Kaggle Drive mode), so output/ and
# result/ live right inside the batch folder.
ON_KAGGLE_DATASET = FOLDER_PATH.startswith("/kaggle/input")
WORK_ROOT = "/kaggle/working" if ON_KAGGLE_DATASET else FOLDER_PATH
RESULT_DIR = os.path.join(WORK_ROOT, "result")
os.makedirs(RESULT_DIR, exist_ok=True)
print("Merged patch files will be written to:", RESULT_DIR)

# Per-model sample rate resolved in Cell 5.
summary = []

for entry in sorted(batch_manifest["patches"], key=lambda e: e["patch_index"]):
    label = f"patch {entry['patch_index']:03d} ({entry['patch_name']})"
    if PATCH_IDS is not None and entry["patch_id"] not in PATCH_IDS:
        summary.append((label, "skipped (not in PATCH_IDS)"))
        continue

    patch_dir = os.path.join(FOLDER_PATH, entry["folder"])
    out_dir = os.path.join(WORK_ROOT, entry["folder"], "output")
    result_path = os.path.join(WORK_ROOT, entry["result_wav"])
    print(f"\n=== {label}: {entry['chunk_count']} chunks ===")
    manifest_path = os.path.join(patch_dir, "manifest.json")
    try:
        with open(manifest_path, "r", encoding="utf-8") as f: manifest = json.load(f)
    except Exception as exc:
        print(f"patch failed - manifest unreadable: {exc}")
        summary.append((label, "failed (manifest unreadable)"))
        continue
    # Compact-manifest support: exports omit the derivable chunks /
    # expected_outputs lists and the per-chunk filename/chapter_title. Rebuilding
    # them here is also the manifest check, so everything below sees one shape.
    if not isinstance(manifest, dict) or not normalize_chunk_manifest(manifest):
        print("patch failed - manifest has no chunks list and no chunk_count")
        summary.append((label, "failed (invalid manifest)"))
        continue

    timeline_path = os.path.splitext(result_path)[0] + ".timeline.json"
    # Result WAVs are never downloaded, so on a restarted Kaggle session the local
    # file is absent even though the patch is finished. Without the REMOTE half every
    # finished patch would be re-merged, dragging its chunk WAVs back down.
    already_merged = os.path.exists(result_path) or entry["result_wav"] in REMOTE
    if SKIP_EXISTING and already_merged:
        timeline_rel = os.path.splitext(entry["result_wav"])[0] + ".timeline.json"
        if manifest.get("chunk_metadata") and not (
            os.path.exists(timeline_path) or timeline_rel in REMOTE
        ):
            warnings.warn("Timeline sidecar missing; delete result and rerun")
        print(f"already merged -> {result_path} (skipping patch)")
        summary.append((label, "done (already merged)"))
        continue

    os.makedirs(out_dir, exist_ok=True)
    remote_out_dir = entry["folder"] + "/output"
    available = available_wavs(
        [out_dir, os.path.join(patch_dir, "output")], REMOTE, remote_out_dir
    )

    for offset, chunk_filename in enumerate(manifest["chunks"]):
        index = chunk_filename.split("_")[1].split(".")[0]  # chunk_000.txt -> 000
        wav_name = f"chunk_{index}.wav"
        if SKIP_EXISTING and wav_name in available:
            print(f"skip {chunk_filename} (already synthesized)")
            continue

        text = chunk_text_for(manifest, offset, patch_dir)
        out_path = os.path.join(out_dir, wav_name)

        if TTS_MODEL in ("edge-tts", "gtts"):
            mp3_tmp = out_path + ".mp3"
            save_online_mp3(text, VOICE_ID, TTS_MODEL, mp3_tmp)
            mp3_to_wav(mp3_tmp, out_path)
        else:
            kwargs = {}
            if TTS_MODEL == "voxcpm2":
                if reference_wav_path:
                    kwargs["reference_wav_path"] = reference_wav_path
                    if prompt_text:
                        kwargs["prompt_wav_path"] = reference_wav_path
                        kwargs["prompt_text"] = prompt_text
                import torch
                torch.manual_seed(42)  # VoxCPM2 no longer accepts a seed= argument
                audio = model.generate(text=text, cfg_value=2.0, inference_timesteps=10, **kwargs)
                sf.write(out_path, audio, SAMPLE_RATE)
            elif TTS_MODEL == "omnivoice":
                if reference_wav_path:
                    kwargs["ref_audio"] = reference_wav_path
                    if prompt_text:
                        kwargs["ref_text"] = prompt_text
                import torch
                torch.manual_seed(42)
                result = model.generate(text=text, **kwargs)
                audio = result[0] if isinstance(result, (list, tuple)) else result
                sf.write(out_path, audio, SAMPLE_RATE)
            elif TTS_MODEL == "vieneu-fast":
                if reference_wav_path:
                    kwargs["ref_audio"] = reference_wav_path
                audio = model.infer(text, style=TTS_OPTIONS.get("style", "doc_truyen"), **kwargs)
                sf.write(out_path, audio, SAMPLE_RATE)
            else:
                raise RuntimeError(f"Unsupported TTS model: {TTS_MODEL}")

        persist(out_path, entry["folder"] + "/output")
        available.add(wav_name)
        print(f"wrote {out_path}")

    # Merge this patch right away (instead of one big merge at the end) so an
    # interrupted batch still yields finished result files for completed patches.
    missing = [w for w in manifest["expected_outputs"] if w not in available]
    if missing:
        print(f"patch incomplete - {len(missing)} chunk(s) missing "
              f"(first: {missing[0]}); re-run this cell to resume")
        summary.append((label, f"incomplete ({len(missing)} chunks missing)"))
        continue

    # Every expected chunk exists locally or on Drive. Pull down only the ones that
    # are not local yet - for a restarted session that is just this patch's earlier
    # chunks, not the whole batch.
    paths, pending = [], []
    for wav_name in manifest["expected_outputs"]:
        local = next(
            (c for c in (os.path.join(out_dir, wav_name),
                         os.path.join(patch_dir, "output", wav_name))
             if os.path.exists(c)),
            None,
        )
        if local:
            paths.append(local)
        else:
            dest = os.path.join(out_dir, wav_name)
            pending.append((remote_out_dir + "/" + wav_name, dest))
            paths.append(dest)

    if pending:
        print(f"fetching {len(pending)} chunk WAV(s) from Drive...")
        try:
            drive_fetch_many(pending)
        except Exception as exc:
            print(f"patch incomplete - chunk download failed: {exc}")
            summary.append((label, "incomplete (chunk download failed)"))
            continue

    try:
        timeline = merge_wav_files(paths, result_path, _CHUNK_PAUSE_MS, manifest.get("chunk_metadata"), manifest["chunks"])
    except Exception as exc:
        print(f"patch failed during WAV merge: {exc}")
        summary.append((label, "failed (WAV merge)"))
        continue
    try: persist(result_path, "result")
    except Exception as exc: warnings.warn(f"Result persistence failed after local install: {exc}")
    if timeline is not None:
        try:
            write_timeline_atomic(timeline, timeline_path)
        except Exception as exc:
            warnings.warn(f"Timeline write failed; existing sidecar preserved: {exc}")
        try: persist(timeline_path, "result")
        except Exception as exc: warnings.warn(f"Timeline persistence failed after local install: {exc}")
    print(f"merged {len(paths)} chunks -> {result_path}")
    summary.append((label, "merged"))

print("\n=== Batch summary ===")
for name, status in summary:
    print(f"- {name}: {status}")
